Establish connection to Oracle database using SQLAlchemy with specified credentials

In [ ]:
import pandas as pd #Imports pandas library for data manipulation and analysis.
from sqlalchemy import create_engine    #Imports SQLAlchemy function to create a connection engine for interacting with databases.

#Define Oracle database connection parameters including username, password, host, port, and service name.
user = "saptest"
password = "Corpdb#5aptest"
host = "corpdb.bhel.in"
port = "1521"
service = "ORCL"

#Creates a SQLAlchemy engine to establish a connection with the Oracle database using provided credentials.
engine = create_engine(
    f"oracle+oracledb://{user}:{password}@{host}:{port}/?service_name={service}"
)

Execute SQL query to retrieve data from the Oracle table and load it into a pandas DataFrame for further analysis.

In [3]:
query = """
SELECT *
FROM SAPTEST.PROJECT_MONTHLY_SNAPSHOT_TB
"""

df = pd.read_sql(query, engine)

Display the first few rows of the DataFrame to quickly inspect the structure and contents of the loaded data.

In [74]:
df.head()

,emp_id,snapshot_month,age_yy,function,cadre,grade,unit,location,tenure_mm,training_days_36m,...,incentive_sum_last3y,resig_flag,resig_same_unit_grade,absence_days_6m,absence_rate_3m,absence_rate_30d,absence_spike,sal_growth,compa_ratio,below_peer_flag
0,1261894,2021-01-01,59.53,50000339,5,68,1010,0002,410.55,1,...,NaN,0,0.0,47.0,0.1556,0.0,0.0,0.0655,1.086211,N
1,1258915,2021-01-01,55.86,50000354,5,68,1010,0002,409.19,0,...,NaN,0,0.0,18.0,0.1000,0.0,0.0,0.0654,1.027485,N
2,1270257,2021-01-01,55.66,50000353,3,48,1010,0002,363.81,0,...,NaN,0,0.0,7.0,0.0778,0.0,0.0,0.0654,1.088938,N
3,1270281,2021-01-01,56.48,50000353,1,00,1010,0002,396.58,0,...,NaN,0,0.0,36.0,0.3333,0.0,0.0,0.0278,1.770718,N
4,6042813,2021-01-01,34.82,50000341,1,05,1010,0002,146.87,3,...,NaN,0,0.0,17.0,0.1000,0.0,0.0,0.0579,0.982744,N


List all column names in the DataFrame to understand the available fields in the dataset.

In [75]:
df.columns

Index(['emp_id', 'snapshot_month', 'age_yy', 'function', 'cadre', 'grade',
       'unit', 'location', 'tenure_mm', 'training_days_36m', 'pos_chng_count',
       'perf_slope', 'curr_score', 'perf_delta', 'months_since_prom',
       'prom_count', 'stagflation', 'incentive_volatility',
       'incentive_sum_last3y', 'resig_flag', 'resig_same_unit_grade',
       'absence_days_6m', 'absence_rate_3m', 'absence_rate_30d',
       'absence_spike', 'sal_growth', 'compa_ratio', 'below_peer_flag'],
      dtype='object')

Calculate the total number of missing (null) values in each column to assess data completeness.

In [76]:
df.isna().sum()

emp_id                        0
snapshot_month                0
age_yy                        0
function                      0
cadre                         0
grade                         0
unit                          0
location                      0
tenure_mm                     0
training_days_36m             0
pos_chng_count                0
perf_slope               375442
curr_score               271843
perf_delta                    0
months_since_prom         32292
prom_count                32292
stagflation               32292
incentive_volatility      32292
incentive_sum_last3y      32292
resig_flag                    0
resig_same_unit_grade      1021
absence_days_6m            1021
absence_rate_3m            1021
absence_rate_30d           1021
absence_spike              1021
sal_growth                 7942
compa_ratio                2775
below_peer_flag            1155
dtype: int64

Perform data cleaning by handling missing values using defaults, medians, and group-wise imputations to ensure completeness and consistency of features for analysis.

In [4]:
#Replacing missing values in Below peer flag with "N"  (If no data → assume employee is not below peer)
df['below_peer_flag'] = df['below_peer_flag'].fillna("N")

#Replacing missing values in Performance related features with 0 (Missing value = absence of event)
df['perf_slope'] = df['perf_slope'].fillna(0)
df['prom_count'] = df['prom_count'].fillna(0)
df['stagflation'] = df['stagflation'].fillna(0)
df['incentive_volatility'] = df['incentive_volatility'].fillna(0)
df['incentive_sum_last3y'] = df['incentive_sum_last3y'].fillna(0)
df['resig_same_unit_grade'] = df['resig_same_unit_grade'].fillna(0)

#Replacing missing values in Absence Features with Median values (Replace missing with “normal absence behavior”)
df['absence_days_6m'] = df['absence_days_6m'].fillna(df['absence_days_6m'].median())
df['absence_rate_3m'] = df['absence_rate_3m'].fillna(df['absence_rate_3m'].median())
df['absence_rate_30d'] = df['absence_rate_30d'].fillna(df['absence_rate_30d'].median())
df['absence_spike'] = df['absence_spike'].fillna(df['absence_spike'].median())

#Replacing missing values in with median values of employees in same grade (Replace with typical value for that grade)
df['sal_growth'] = df.groupby('grade')['sal_growth'].transform(
    lambda x: x.fillna(x.median())
)
df['compa_ratio'] = df.groupby('grade')['compa_ratio'].transform(
    lambda x: x.fillna(x.median())
)
df['curr_score'] = df['curr_score'].fillna(
    df.groupby('grade')['curr_score'].transform('median')
)

#Replacing missing values with Tenure
df['months_since_prom'] = df['months_since_prom'].fillna(df['tenure_mm'])

c:\Users\asad\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\asad\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\asad\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\asad\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\asad\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keep

In [5]:
df.isna().sum()

emp_id                      0
snapshot_month              0
age_yy                      0
function                    0
cadre                       0
grade                       0
unit                        0
location                    0
tenure_mm                   0
training_days_36m           0
pos_chng_count              0
perf_slope                  0
curr_score               1806
perf_delta                  0
months_since_prom           0
prom_count                  0
stagflation                 0
incentive_volatility        0
incentive_sum_last3y        0
resig_flag                  0
resig_same_unit_grade       0
absence_days_6m             0
absence_rate_3m             0
absence_rate_30d            0
absence_spike               0
sal_growth                170
compa_ratio               120
below_peer_flag             0
dtype: int64

Handling residual Missing Values

In [6]:
df['sal_growth'] = df['sal_growth'].fillna(df['sal_growth'].median())
df['compa_ratio'] = df['compa_ratio'].fillna(1)
df['curr_score'] = df['curr_score'].fillna(0)

Recheck Missing/NA Values in the Dataframe

In [7]:
df.isna().sum()

emp_id                   0
snapshot_month           0
age_yy                   0
function                 0
cadre                    0
grade                    0
unit                     0
location                 0
tenure_mm                0
training_days_36m        0
pos_chng_count           0
perf_slope               0
curr_score               0
perf_delta               0
months_since_prom        0
prom_count               0
stagflation              0
incentive_volatility     0
incentive_sum_last3y     0
resig_flag               0
resig_same_unit_grade    0
absence_days_6m          0
absence_rate_3m          0
absence_rate_30d         0
absence_spike            0
sal_growth               0
compa_ratio              0
below_peer_flag          0
dtype: int64

In [8]:
df.head()

,emp_id,snapshot_month,age_yy,function,cadre,grade,unit,location,tenure_mm,training_days_36m,...,incentive_sum_last3y,resig_flag,resig_same_unit_grade,absence_days_6m,absence_rate_3m,absence_rate_30d,absence_spike,sal_growth,compa_ratio,below_peer_flag
0,1261894,2021-01-01,59.53,50000339,5,68,1010,0002,410.55,1,...,0.0,0,0.0,47.0,0.1556,0.0,0.0,0.0655,1.086211,N
1,1258915,2021-01-01,55.86,50000354,5,68,1010,0002,409.19,0,...,0.0,0,0.0,18.0,0.1000,0.0,0.0,0.0654,1.027485,N
2,1270257,2021-01-01,55.66,50000353,3,48,1010,0002,363.81,0,...,0.0,0,0.0,7.0,0.0778,0.0,0.0,0.0654,1.088938,N
3,1270281,2021-01-01,56.48,50000353,1,00,1010,0002,396.58,0,...,0.0,0,0.0,36.0,0.3333,0.0,0.0,0.0278,1.770718,N
4,6042813,2021-01-01,34.82,50000341,1,05,1010,0002,146.87,3,...,0.0,0,0.0,17.0,0.1000,0.0,0.0,0.0579,0.982744,N


Converting Colum Headers to Upper Case

In [9]:
df.columns = df.columns.str.upper()

Standardizes the GRADE column by converting non-numeric values to numeric format.

In [10]:
df["GRADE"] = df["GRADE"].astype(str)
df["GRADE"] = df["GRADE"].str.replace("T", "9")
df["GRADE"] = df["GRADE"].str.replace("D", "9")
df["GRADE"] = df["GRADE"].astype(int)

Converts the BELOW_PEER_FLAG column from categorical (Y/N) to numeric (1/0) format

In [11]:
df['BELOW_PEER_FLAG'].value_counts()

BELOW_PEER_FLAG
N    1374029
Y     388951
Name: count, dtype: int64

In [12]:
df['BELOW_PEER_FLAG'] = df['BELOW_PEER_FLAG'].map({"Y":1,"N":0})

Convert categorical columns into dummy variables for machine learning.

In [13]:
df = pd.get_dummies(df, columns=["UNIT","FUNCTION","CADRE","GRADE"], drop_first=True)

Standardize LOCATION by converting to string and replacing non-numeric values.

In [14]:
df['LOCATION'] = df['LOCATION'].astype(str)
df['LOCATION'] = df['LOCATION'].str.replace("F","9")

Identify highly correlated features (>0.8) to remove multicollinearity in the dataset.

In [15]:
import pandas as pd
import numpy as np

# correlation matrix
corr_matrix = df.corr().abs()

# select upper triangle of correlation matrix
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# find columns with correlation > threshold
threshold = 0.8

to_drop = [
    column for column in upper.columns
    if any(upper[column] > threshold)
]

print("Columns to drop:", to_drop)

Columns to drop: ['PROM_COUNT', 'ABSENCE_RATE_3M', 'ABSENCE_RATE_30D', 'GRADE_93', 'GRADE_95']


Drop highly correlated features from the Dataframe

In [16]:
df = df.drop(columns=['PROM_COUNT', 'ABSENCE_RATE_3M', 'ABSENCE_RATE_30D', 'GRADE_93', 'GRADE_95'])

Fetch Latest Dataframe (Snapshot Date = 2025-12-01) & Training Dataframe (Snapshot Date between 01-01-2021 to 01-07-2025)

In [17]:
latest_snapshot = pd.Timestamp("2025-12-01")

df_latest = df[df["SNAPSHOT_MONTH"] == latest_snapshot].copy()

df_train = df[(df["SNAPSHOT_MONTH"] <= "2025-07-01") & (df["SNAPSHOT_MONTH"] >= "2021-01-01")].copy()

Define target variable for model training.

In [18]:
y = df_train["RESIG_FLAG"]

Remove non-feature columns to create input datasets for model training and prediction.

In [19]:
drop_cols = [
    "EMP_ID",
    "SNAPSHOT_MONTH",
    "RESIG_FLAG",
    "LOCATION"
]

X = df_train.drop(columns=drop_cols, errors="ignore")
X_latest = df_latest.drop(columns=drop_cols, errors="ignore")

Align latest dataset columns with training data, filling missing columns with zeros.

In [20]:
X_latest = X_latest.reindex(columns=X.columns, fill_value=0)

Import libraries for data handling, database access, model training, and evaluation.

In [21]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import roc_auc_score

Split data into training and test sets while preserving class distribution (Train = 75% / Test = 25%). 

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42, #Controls randomness
    stratify=y #Ensures class balance is preserved
)

NameError: name 'train_test_split' is not defined

Create a pipeline with scaling and logistic regression model for balanced classification.

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()), #standardizes features
    ("model", LogisticRegression(  #create Logistic Regression Model
        class_weight="balanced",
        max_iter=2000,
        C=0.5
    ))
])

Train the model pipeline using the training data.

In [24]:
pipe.fit(X_train, y_train)

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,0.5


Generate predicted probabilities for test data.

In [25]:
pred_test = pipe.predict_proba(X_test)[:,1]

Calculate ROC AUC score to evaluate model performance.

In [26]:
auc = roc_auc_score(y_test, pred_test)

print("ROC AUC Score:", auc)

ROC AUC Score: 0.9655187682240609


Generate predicted attrition probabilities for the latest dataset.

In [27]:
prob_latest = pipe.predict_proba(X_latest)[:,1]
df_latest["RESIG_PROBAB"] = prob_latest

Define database credentials and create Oracle connection engine for data access.

In [28]:
user = "saptest"
password = "Corpdb#5aptest"
host = "corpdb.bhel.in"
port = "1521"
service = "ORCL"

engine = create_engine(
    f"oracle+oracledb://{user}:{password}@{host}:{port}/?service_name={service}"
)

In [29]:
df_latest.head()

,EMP_ID,SNAPSHOT_MONTH,AGE_YY,LOCATION,TENURE_MM,TRAINING_DAYS_36M,POS_CHNG_COUNT,PERF_SLOPE,CURR_SCORE,PERF_DELTA,...,GRADE_70,GRADE_71,GRADE_72,GRADE_73,GRADE_74,GRADE_75,GRADE_94,GRADE_96,GRADE_98,RESIG_PROBAB
135274,3786595,2025-12-01,56.75,0001,420.06,1,1,0.0,100.0,0.0,...,False,False,False,False,False,False,False,False,False,0.008026
135275,6046045,2025-12-01,38.42,0027,205.84,0,0,0.0,85.0,0.0,...,False,False,False,False,False,False,False,False,False,0.210472
135276,6050263,2025-12-01,42.49,0002,205.45,1,1,0.0,98.8,0.0,...,False,False,False,False,False,False,False,False,False,0.000002
135277,6049206,2025-12-01,40.66,0027,205.55,0,0,0.0,100.0,0.0,...,False,False,False,False,False,False,False,False,False,0.003228
135278,6057330,2025-12-01,41.46,0027,204.87,3,0,0.0,100.0,0.0,...,False,False,False,False,False,False,False,False,False,0.004868


Write predicted results to Oracle table in batches for efficient database storage.

In [30]:
df_latest3 = df_latest[['EMP_ID', 'SNAPSHOT_MONTH', 'AGE_YY', 'LOCATION', 'TENURE_MM', 'TRAINING_DAYS_36M', 'POS_CHNG_COUNT', 'PERF_SLOPE', 'CURR_SCORE', 'PERF_DELTA', 'MONTHS_SINCE_PROM', 'STAGFLATION', 'INCENTIVE_VOLATILITY', 'INCENTIVE_SUM_LAST3Y', 'RESIG_FLAG', 'RESIG_SAME_UNIT_GRADE', 'ABSENCE_DAYS_6M', 'ABSENCE_SPIKE', 'SAL_GROWTH', 'COMPA_RATIO', 'BELOW_PEER_FLAG', 'FUNCTION_50000362'
,'GRADE_27','GRADE_36','GRADE_40','GRADE_9','CADRE_6','GRADE_45','GRADE_48','GRADE_24','GRADE_63','GRADE_50','UNIT_1048','GRADE_46','CADRE_5','CADRE_3','UNIT_1014','GRADE_11','GRADE_7','GRADE_22','FUNCTION_50000342','UNIT_1013','UNIT_1032','FUNCTION_50000351','RESIG_PROBAB']]

In [31]:
df_latest3.to_sql(
    name="PROJECT_RESIG_PROBAB",
    schema="SAPTEST",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=5000
)

C:\Users\asad\AppData\Local\Temp\ipykernel_22032\4205703194.py:1: UserWarning: The provided table name 'PROJECT_RESIG_PROBAB' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df_latest3.to_sql(


-6

Extract model coefficients and map them to features for interpretability.

In [32]:
import pandas as pd
import numpy as np

# Extract trained logistic regression model
model = pipe.named_steps["model"]

# Create dataframe of coefficients
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

print(coef_df.head())

             Feature  Coefficient
0             AGE_YY    -1.115314
1          TENURE_MM    -0.972805
2  TRAINING_DAYS_36M     0.089675
3     POS_CHNG_COUNT     0.114090
4         PERF_SLOPE     0.130825


Save feature coefficients to Oracle table for attrition driver analysis.

In [33]:
user = "saptest"
password = "Corpdb#5aptest"
host = "corpdb.bhel.in"
port = "1521"
service = "ORCL"

engine = create_engine(
    f"oracle+oracledb://{user}:{password}@{host}:{port}/?service_name={service}"
)

In [34]:
coef_df.columns = coef_df.columns.str.upper()

In [35]:
coef_df.to_sql(
    name="PROJECT_ATTRITION_DRIVERS",
    schema="SAPTEST",
    con=engine,
    if_exists="append",
    index=False
)

C:\Users\asad\AppData\Local\Temp\ipykernel_22032\2600310558.py:1: UserWarning: The provided table name 'PROJECT_ATTRITION_DRIVERS' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  coef_df.to_sql(


-1